In [1]:
from pyiceberg.catalog import load_catalog
from pyiceberg.schema import Schema
from pyiceberg.types import NestedField, LongType, StringType, DoubleType
import pyarrow as pa
from pprint import pprint

In [2]:
catalog = load_catalog("default")
print(catalog.list_namespaces())

[('open_lakehouse',)]


In [11]:
%pip install --upgrade "aiobotocore>=3.7.0" boto3 botocore s3transfer

INFO: pip is looking at multiple versions of boto3 to determine which version is compatible with other requirements. This could take a while.
  Using cached boto3-1.43.47-py3-none-any.whl.metadata (6.6 kB)
INFO: pip is still looking at multiple versions of boto3 to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.5/140.5 kB 645.2 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.0/15.0 MB 5.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.3/88.3 kB 6.5 MB/s eta 0:00:00
  Attempting uninstall: botocore
    Found existing installation: botocore 1.43.47
    Uninstalling botocore-1.43.47:
      Successfully uninstalled

In [5]:
import boto3, json

s3 = boto3.client(
    "s3",
    endpoint_url="http://minio:9000",
    aws_access_key_id="minioadmin",
    aws_secret_access_key="minioadmin",
)

# List everything under the table prefix
response = s3.list_objects_v2(Bucket="warehouse")

print(response)

{'ResponseMetadata': {'RequestId': '18C2380D95A7F1C9', 'HostId': 'dd9025bab4ad464b049177c95eb6ebf374d3b3fd1af9251148b658df7ac2e3e8', 'HTTPStatusCode': 200, 'HTTPHeaders': {'accept-ranges': 'bytes', 'content-length': '564', 'content-type': 'application/xml', 'server': 'MinIO', 'strict-transport-security': 'max-age=31536000; includeSubDomains', 'vary': 'Origin, Accept-Encoding', 'x-amz-id-2': 'dd9025bab4ad464b049177c95eb6ebf374d3b3fd1af9251148b658df7ac2e3e8', 'x-amz-request-id': '18C2380D95A7F1C9', 'x-content-type-options': 'nosniff', 'x-ratelimit-limit': '3355', 'x-ratelimit-remaining': '3355', 'x-xss-protection': '1; mode=block', 'date': 'Tue, 14 Jul 2026 17:27:13 GMT'}, 'RetryAttempts': 0}, 'IsTruncated': False, 'Contents': [{'Key': 'open_lakehouse/yellow_taxi_trips/metadata/00000-5efeec3b-fa4e-44c8-8c1e-3d24612ea73d.metadata.json', 'LastModified': datetime.datetime(2026, 7, 14, 15, 58, 17, 676000, tzinfo=tzlocal()), 'ETag': '"5ae8b82d0df0f1fbaf5c3a4e645a6dd2"', 'Size': 1944, 'Storage

In [13]:
import duckdb
from pyiceberg.catalog import load_catalog


# 1. Load the REST Catalog
catalog = load_catalog(
    "default"
)

# 2. Load the Table from the Catalog
table_identifier = "open_lakehouse.yellow_taxi_trips"
table = catalog.load_table(table_identifier)

# print(f"📖 Loaded Iceberg Table: {table_identifier}")
# print(f"Schema fields: {[field.name for field in table.schema().fields]}")

# DuckDB can query the PyArrow Table in-memory instantly
arrow_table = table.scan().to_arrow()

result = duckdb.query("""
    SELECT 
        *
    FROM arrow_table
""").to_df()

print(result)

Empty DataFrame
Columns: [VendorID, tpep_pickup_datetime, tpep_dropoff_datetime, passenger_count, trip_distance, RatecodeID, store_and_fwd_flag, PULocationID, DOLocationID, payment_type, fare_amount, extra, mta_tax, tip_amount, tolls_amount, improvement_surcharge, total_amount, congestion_surcharge, Airport_fee, cbd_congestion_fee]
Index: []


In [14]:
catalog.drop_table("open_lakehouse.yellow_taxi_trips")